# 1 — Reformat: CFD → Brackish frames on the VM disk

**Question.** Does a ~3.4M-parameter CenterNet-style head on *frozen* DINOv3 ConvNeXt-B features detect fish competitively on the Brackish source of the Community Fish Detection Dataset (CFD), scored by the same `pycocotools` harness as the released RF-DETR baselines on the *identical* val images?

**Layout — nothing touches the laptop.**

| Location | Holds |
|---|---|
| `/content/` (VM SSD, ephemeral) | CFD metadata, the Brackish images (re-fetched each session — cheap on the LILA/GCS pipe) |
| Google Drive `frozen-trunk-detection/` | converted DINOv3 backbone weights (`weights/`), run dirs (`runs/<name>/` — `best.pt`, `history.json`, `curves.png`, `predictions.json`), `results/` (manifest, baseline predictions, `results.csv`, viz) |

**Do not put images on Drive** — per-file Drive API latency makes 12k JPEG reads crawl.

**Runtime.** T4/L4 is enough here: only the head trains and the backbone runs under `no_grad`, so the run is very likely CPU-bound on JPEG decode + augmentation. The throughput probe in `2_training` measures GPU utilisation before you spend on anything bigger. If the VM dies, re-run `1_reformat` (minutes) — every artefact of value is already on Drive.

**Rules written before the numbers** (also in [`docs/report.md`](docs/report.md)): never compare against the README's `.609` (different split — we re-run their weights instead); the baselines likely *saw* these val images, so their number is biased in their favour — our win is conservative, our loss is ambiguous and is reported as ambiguous; report trainable params **and** total inference FLOPs.

---

**This notebook (1 of 4)** turns the CFD master metadata into a per-source manifest, subsets the Brackish source on the published `is_train` split, and fetches its frames to the VM disk resized to long side 1024.

## 1 · Drive, paths, code

Mounts Drive, fixes the four paths every cell below uses, clones the branch and installs it editable. Safe to re-run: the clone is wiped and redone each time.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, shutil, subprocess, pathlib
DRIVE = '/content/drive/MyDrive/frozen-trunk-detection'
for sub in ('weights', 'runs', 'results', 'results/manifest', 'results/viz'):
    os.makedirs(f'{DRIVE}/{sub}', exist_ok=True)
REPO = '/content/crop-counter'
DATA = '/content/data/brackish'
CFD  = '/content/cfd'
os.makedirs(CFD, exist_ok=True)
print(os.listdir(DRIVE))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/detection-head --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
# torch/torchvision come with the Colab image. torchmetrics + termcolor are needed only so
# Meta's dinov3 hubconf imports (it pulls the segmentors); pycocotools for COCO AP; ijson to
# stream the 1.9M-record CFD metadata without json.load-ing it.
!pip install -q -e ".[detection]" ijson
# A running kernel does not re-read site-packages' .pth files, so the editable install is invisible
# to THIS process until restart (subprocess calls like `!python -m cropcounter.train` see it fine).
import sys, importlib
if '/content/crop-counter/src' not in sys.path:
    sys.path.insert(0, '/content/crop-counter/src')
importlib.invalidate_caches()
import cropcounter, torch
print('cropcounter', cropcounter.__file__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2 · CFD metadata → per-source manifest

47 MB zipped; the JSON inside is streamed (`ijson`), never `json.load`-ed. The manifest is the source for every subsetting decision in the memo (empty-image fraction, `is_train` balance, box-size percentiles, stride-4 centre-cell collision rate, licence).

In [ ]:
META = f'{CFD}/community_fish_detection_dataset.json.zip'
if not os.path.exists(META):
    !wget -q --show-progress -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
t0 = time.time()
!python -m cropcounter.cfd manifest --metadata {META} --out {CFD}/manifest
print(f'manifest in {time.time()-t0:.0f}s')
for f in os.listdir(f'{CFD}/manifest'):
    shutil.copy(f'{CFD}/manifest/{f}', f'{DRIVE}/results/manifest/{f}')

## 3 · Subset — all of Brackish, honouring the published `is_train` split

In [ ]:
import csv, json
# Resolve the exact `dataset` field string for Brackish from the manifest rather than hard-coding it.
rows = list(csv.DictReader(open(f'{CFD}/manifest/manifest.csv')))
name_col = next(c for c in rows[0].keys() if c.lower() in ('dataset', 'source', 'name'))
BRACKISH_SOURCE = next(r[name_col] for r in rows if 'rackish' in r[name_col])
print('Brackish source string:', repr(BRACKISH_SOURCE))
!rm -rf {DATA}
!python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources "{BRACKISH_SOURCE}" --train-cap 100000 --val-cap 100000 --seed 0
print(json.dumps(json.load(open(f'{DATA}/subset_summary.json')), indent=1)[:2000])
!wc -l {DATA}/download_list.txt

## 4 · Fetch images to the VM disk, resized to long side 1024 on write

Resize at fetch, not at train: (i) fairness — the released baselines run at 1024/640 and AP is acutely resolution-sensitive for small objects; (ii) train/val scale consistency; (iii) disk. Boxes and `width`/`height` are rescaled into `annotations.json`; the native-scale file is kept as `annotations.native.json`.

In [ ]:
t0 = time.time()
!python -m cropcounter.cfd fetch --subset {DATA} --max-side 1024 --workers 32 --mirror gcs
print(f'fetched in {time.time()-t0:.0f}s')
!du -sh {DATA}/train/images {DATA}/val/images
!ls {DATA}/train/images | wc -l; ls {DATA}/val/images | wc -l

## What the data looks like

A seeded sample of frames per split (two thirds drawn from frames that contain fish, the rest empty — 60 % of Brackish frames are empty), ground-truth boxes in green, plus the per-frame count, box-size and aspect distributions. Saved to Drive `results/viz/` and reused in the report.

In [ ]:
import json, random, os, numpy as np
import matplotlib; matplotlib.use('Agg')
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from PIL import Image
from IPython.display import Image as IPImage, display
os.makedirs(f'{DRIVE}/results/viz', exist_ok=True)

def show_samples(split, n=12, seed=0):
    ann = json.load(open(f'{DATA}/{split}/annotations.json'))
    by_img = {}
    for a in ann['annotations']:
        by_img.setdefault(a['image_id'], []).append(a['bbox'])
    imgs = ann['images']; rng = random.Random(seed)
    pos = [im for im in imgs if by_img.get(im['id'])]; neg = [im for im in imgs if not by_img.get(im['id'])]
    k_pos = min(n * 2 // 3, len(pos)); picks = rng.sample(pos, k_pos) + rng.sample(neg, min(n - k_pos, len(neg)))
    cols = 4; rows = -(-len(picks) // cols)
    fig = Figure(figsize=(4.2 * cols, 2.7 * rows)); FigureCanvasAgg(fig)
    axes = np.atleast_1d(fig.subplots(rows, cols)).ravel()
    for ax, im in zip(axes, picks):
        ax.imshow(Image.open(f"{DATA}/{split}/images/{im['file_name']}"))
        for x, y, w, h in by_img.get(im['id'], []):
            ax.add_patch(matplotlib.patches.Rectangle((x, y), w, h, fill=False, edgecolor='lime', linewidth=1.2))
        seq = im.get('cfd_sequence') or im.get('original_data_source') or ''
        ax.set_title(f"{split} · {len(by_img.get(im['id'], []))} fish · {str(seq)[:30]}", fontsize=8); ax.axis('off')
    for ax in axes[len(picks):]:
        ax.axis('off')
    n_boxes = sum(len(v) for v in by_img.values())
    fig.suptitle(f"Brackish {split}: {len(imgs):,} frames · {n_boxes:,} boxes · {100 * len(neg) / max(len(imgs), 1):.0f}% empty — ground-truth boxes in green (2/3 of the sample drawn from frames with fish)", fontsize=11)
    out = f'{DRIVE}/results/viz/data_samples_{split}.png'; fig.savefig(out, dpi=100, bbox_inches='tight'); display(IPImage(out)); return out

show_samples('train'); show_samples('val')

fig = Figure(figsize=(14, 3.6)); FigureCanvasAgg(fig); ax1, ax2, ax3 = fig.subplots(1, 3)
for split, col in (('train', '#2b5f9e'), ('val', '#c1440e')):
    ann = json.load(open(f'{DATA}/{split}/annotations.json'))
    per = {}
    for a in ann['annotations']:
        per[a['image_id']] = per.get(a['image_id'], 0) + 1
    counts = np.array([per.get(im['id'], 0) for im in ann['images']])
    ws = np.array([a['bbox'][2] for a in ann['annotations']], dtype=float); hs = np.array([a['bbox'][3] for a in ann['annotations']], dtype=float)
    ax1.hist(counts, bins=np.arange(0, counts.max() + 2) - 0.5, alpha=.6, color=col, label=f'{split} (n={len(counts):,})', density=True)
    ax2.hist(np.sqrt(ws * hs), bins=40, alpha=.6, color=col, label=split, density=True)
    ax3.scatter(ws, hs, s=3, alpha=.25, color=col, label=split)
ax1.set(title='fish per frame', xlabel='boxes in frame', ylabel='fraction of frames'); ax1.legend(fontsize=8)
ax2.set(title='box size √(w·h), px', xlabel='px'); ax2.axvline(4, color='k', ls=':', lw=1); ax2.legend(fontsize=8)
ax3.set(title='box w × h, px (log)', xlabel='w', ylabel='h', xscale='log', yscale='log'); ax3.legend(fontsize=8)
out = f'{DRIVE}/results/viz/data_stats.png'; fig.savefig(out, dpi=110, bbox_inches='tight'); display(IPImage(out))


## 5 · Next

`2_training.ipynb` — backbone metric-reproduction check, throughput probe, the frozen go/no-go run and the linear probe. Keep this VM alive: the frames just fetched are on its disk.